# Task 1 — Article Type Classification

**Status:** Sections 1–7 define the task data, evaluation policy, and preprocessing plan. There is no
trained model, fake result, winning-model choice, or final claim in this notebook.

**Owner:** TODO(owner)


## 1. Task contract

Task 1 predicts the `articleType` of one fashion product from its image.

- **Input:** one fashion image from the provided dataset identified by its product `id`.
- **Prediction unit:** one product image.
- **Output:** one label from the fixed 124-class `articleType` vocabulary.
- **Final use:** the prediction will fill the `articleType` column in the final
  `id,gender,articleType,season,usage` submission file.
- **Development scope:** compare models using labelled development images and the
  five saved cross-validation folds.
- **Out of scope:** opening holdout labels, creating another split, using external
  images, or selecting the final submitted model before experiments are complete.
- **Main risks:** rare classes, visually similar article types, small images and
  possible product-family leakage.

The final submitted learned model will be trained from scratch. Pretrained weights
may only be used later as a clearly separated comparison benchmark.


## 2. Reading the data

Task 1 uses the teacher-image rows recorded in the shared manifest:

`data/processed/splits.csv`

The manifest must be loaded through `fashion.data.dataset.load_splits`. This keeps
holdout and quarantine labels hidden.

For Task 1, we keep rows that:

- belong to the `development` partition;
- have a valid `articleType` label;
- have an image path;
- use one of the 124 development-defined article-type labels.

We do not call `train_test_split`. The five `cv_fold` values created by Notebook 01
are the only allowed development splits.

This section loads metadata and image paths. It does not load image pixels yet.
Image decoding, resizing and normalization will be added after the preprocessing
comparison is defined.


In [1]:
from fashion.data.dataset import (
    get_samples,
    iter_cv_folds,
    load_label_maps,
    load_splits,
)

TARGET = "articleType"

# Safe loader: protected holdout and quarantine labels remain hidden.
splits = load_splits()

# Labelled Task 1 development rows only.
task1_development = get_samples(
    splits,
    partition="development",
    target=TARGET,
)

# Vocabulary created from development data by Notebook 01.
article_type_map = load_label_maps()[TARGET]
article_type_classes = tuple(article_type_map["classes"])
label_to_index = article_type_map["label_to_index"]

NUM_CLASSES = len(article_type_classes)


In [2]:
protected = splits["partition"].isin(["holdout", "quarantine"])

assert set(task1_development["partition"]) == {"development"}
assert task1_development["has_articleType_label"].all()
assert task1_development["id"].is_unique
assert task1_development["path"].astype(str).str.strip().ne("").all()
assert splits.loc[protected, TARGET].eq("").all()
assert NUM_CLASSES == 124
assert set(task1_development[TARGET]) == set(article_type_classes)

print(f"Task 1 development products: {len(task1_development):,}")
print(f"Article-type classes: {NUM_CLASSES}")
print("Protected labels remain sealed.")


Task 1 development products: 32,773
Article-type classes: 124
Protected labels remain sealed.


In [3]:
import pandas as pd

fold_rows = []

for fold, training_rows, validation_rows in iter_cv_folds(splits):
    training_rows = get_samples(training_rows, target=TARGET)
    validation_rows = get_samples(validation_rows, target=TARGET)

    fold_rows.append(
        {
            "validation_fold": fold,
            "training_products": len(training_rows),
            "validation_products": len(validation_rows),
            "training_classes": training_rows[TARGET].nunique(),
            "validation_classes": validation_rows[TARGET].nunique(),
        }
    )

fold_summary = pd.DataFrame(fold_rows)
fold_summary


,validation_fold,training_products,validation_products,training_classes,validation_classes
0,0,26220,6553,123,102
1,1,26217,6556,120,108
2,2,26220,6553,122,111
3,3,26219,6554,122,107
4,4,26216,6557,121,109


## 3. Development-validation strategy

Every Task 1 model will use all five saved folds from
`data/processed/splits.csv`.

For each round:

1. Four development folds are used for training.
2. The remaining fold is used for validation.
3. The process repeats until every fold has been used for validation once.

The saved fold assignments will not be changed. Rows without a valid
`articleType` label are removed using `has_articleType_label`.

Preprocessing values must be learned from the current training folds only.
Holdout and quarantine rows are not used during model development. The holdout
stays sealed until Notebook 06.


## 4. Fold results and aggregation

Every comparable model must use folds 0, 1, 2, 3, and 4. We will not select or
report only the best-looking fold.

The main result for one model is the arithmetic mean of its five fold-level
primary scores.

We will also report the sample standard deviation using `ddof=1`. This shows
how stable the model is across the five folds.

Every eligible development product must receive exactly one validation
prediction across the five rounds.

Metrics calculated from all out-of-fold predictions may support analysis, but
they will not replace the mean of the five fold scores when models are ranked.


## 5. Metric selection

The primary development metric is strict fixed-label macro-F1 across all 124
`articleType` classes, using `zero_division=0`.

Macro-F1 gives every class equal importance. This is suitable because Task 1
has common classes with many products and rare classes with very few products.
Accuracy alone could hide poor results on rare classes.

Models will be compared using:

- Mean macro-F1 across the five folds.
- Sample standard deviation of macro-F1 across the five folds.

Supporting metrics are:

- Weighted F1.
- Top-1 accuracy.
- Top-5 accuracy.
- Per-class precision, recall, F1, and support.
- One confusion matrix built from all out-of-fold predictions.

A class with no true or predicted validation examples in a fold contributes
zero to that fold's macro-F1. This keeps all 124 classes visible, but it can
make the score sensitive to rare classes missing from a training fold. We will
report this as a limitation instead of changing the saved folds or removing
difficult classes.

This metric policy is frozen before comparing the main HOG and CNN candidates.
It will not be changed because of later results.


## 6. Task-specific preprocessing and leakage rules

### 6.1 EDA evidence behind the pipeline

Notebook 01 found 32,773 image-backed development products, including 294
grayscale images and 12 images that are not the usual 60 x 80 pixels. The
images are small and mostly bright. Stretching changes product shape, while a
centre crop can remove edge details. Exact and near duplicates also exist, so
the saved family-safe folds must not be replaced.

Article Type has 124 classes with product counts from 1 to 5,748. Twenty-six
classes have fold-support warnings, and 12 are untrainable in at least one fold.
Image transforms cannot create missing class evidence, so every class stays in
evaluation and these fold limits will be reported.

### 6.2 Option B preprocessing actions

Only image pixels are available to the model. Year, file size, product name and
the other targets are excluded because they may act as catalogue shortcuts.

For each fold, the pipeline will:

1. Load only the four training folds supplied by `iter_cv_folds`.
2. Apply EXIF orientation, convert every image to RGB, preserve its shape, and
   fit it inside a white 60 x 80 canvas.
3. Fit per-channel RGB mean and standard deviation on those training rows only,
   passing the held-out fold number so accidental validation rows are rejected.
   Artificial padding pixels are excluded from this fit.
4. Apply seeded mild training changes: 50% horizontal flip, up to 5 degrees of
   rotation, 5% movement, 0.95-1.05 scale, and 10% brightness/contrast change.
   The run seed, epoch, and image name make each change repeatable and independent
   of data-loader order.
5. Convert the result to a float32 channel-first array and normalize it.

Validation uses only EXIF orientation, RGB conversion, shape-preserving white
padding, conversion, and the current training-fold normalization. It receives no
random changes and never refits values. Holdout and quarantine rows are rejected
by the normalization fitter and stay sealed until Notebook 06.


In [ ]:
from fashion.task1.preprocessing import (
    DEFAULT_TASK1_PREPROCESSING,
    TASK1_CONTROL_PREPROCESSING,
)

# Declares both controlled alternatives without fitting statistics or training a model.
preprocessing_candidates = {
    "A_no_augmentation": TASK1_CONTROL_PREPROCESSING.to_dict(),
    "B_mild_augmentation": DEFAULT_TASK1_PREPROCESSING.to_dict(),
}
preprocessing_candidates


## 7. Preprocessing comparisons to run

Option B is the EDA-based recommendation, not a declared winner. A controlled
five-fold comparison must show whether its random changes help.

| Comparison ID | Question | Alternative A | Alternative B | Controlled variables | Evidence needed |
|---|---|---|---|---|---|
| `task1-preprocess-augmentation` | Does mild augmentation improve generalisation from tiny catalogue images? | RGB, shape-preserving 60 x 80 white padding, fold-fitted normalization, no random changes | Same deterministic steps plus the seeded mild Option B changes | Same folds, seed, model, loss, optimizer, epochs and early-stopping rule | Mean and standard deviation of five-fold macro-F1, rare-class F1, training time and representative errors |

The chosen preprocessing ID will be recorded in `results/runs.csv`. A later
class-weighting comparison must be a separate experiment so its effect is not
mixed with the augmentation result.


## 8. Hypotheses and baseline

- Baseline purpose and implementation: TODO(owner)
- Hypothesis 1 and expected observation: TODO(owner)
- Hypothesis 2 and expected observation: TODO(owner)
- Long-tail taxonomy and imbalance question: TODO(owner)
- Sampling or loss-weighting options worth comparing: TODO(owner)
- Failure condition that would reject each hypothesis: TODO(owner)

A baseline is a comparison anchor, not a preselected winner.


## 9. Candidate model comparisons

Define candidate families only after checking the assignment constraints.

| Candidate ID | Why include it | Capacity/complexity control | Scratch-training compliance | Expected trade-off |
|---|---|---|---|---|
| TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) |

No candidate is selected in advance.


## 10. Experiment matrix

| Run intent | CV mode/fold | Preprocessing ID | Candidate ID | Controlled seed/budget | Question answered |
|---|---|---|---|---|---|
| TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) |

Keep the matrix broad enough to justify choices, but small enough that each run
receives error and cost analysis.


## 11. Run registry

Every training/evaluation run must append through `fashion.train.registry` to
`results/runs.csv`.

- Run IDs: TODO(owner after execution)
- Checkpoint paths: TODO(owner after execution)
- Configuration hashes: TODO(owner after execution)
- Evidence tables/figures linked to run IDs: TODO(owner after execution)

Do not hand-type final comparison numbers into the report.


## 12. Error analysis

- Define useful error slices before viewing results: TODO(owner)
- Inspect representative successes and failures with IDs: TODO(owner)
- Check rare/ambiguous groups and family effects: TODO(owner)
- Define rare-class error slices before viewing results: TODO(owner)
- Separate data limitations from model limitations: TODO(owner)
- Record unexpected failure modes honestly: TODO(owner)


## 13. Robustness and efficiency

- Robustness questions and controlled tests: TODO(owner)
- Runtime and hardware measurement rule: TODO(owner)
- Memory/storage or index cost: TODO(owner)
- Stability across seeds/folds where applicable: TODO(owner)
- Practical deployment limitation: TODO(owner)


## 14. Decision log

| Decision | Evidence considered | Choice | Rejected alternatives | Limitation | Date/owner |
|---|---|---|---|---|---|
| TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) | TODO(owner) |

Record choices as they are made. Do not rewrite history after holdout access.


## 15. Handoff to final evaluation

Before Notebook 06, provide:

- frozen winning run ID(s): TODO(owner)
- frozen preprocessing configuration: TODO(owner)
- frozen metric definition and CV evidence: TODO(owner)
- refit procedure for all development: TODO(owner)
- expected final checkpoint/output path: TODO(owner)
- unresolved risks and honest limitations: TODO(owner)

**Handoff status: NOT READY — owner must complete every item above.**
